In [1]:

import sys
import itertools
from tqdm.auto import tqdm
import pathlib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

import datasets
from contextlib import nullcontext
import torch
from torch import nn
from transformers import (
    Trainer,
    TrainingArguments,
    LlamaTokenizer,
    LlamaForSequenceClassification,
    TrainerCallback,
    default_data_collator,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    prepare_model_for_int8_training,
)


sys.path.append("../src")
sys.path.append("../config")
from utils import number_split, create_mix


from process_HateSpeech import load_HateSpeech_dynGen, load_HateSpeech_wsf
from process_SHAC import load_process_SHAC



In [2]:
df_dynGen = load_HateSpeech_dynGen()
df_wsf = load_HateSpeech_wsf()

In [2]:
df_shac = load_process_SHAC(replaceNA="all")

df_shac_uw = df_shac.query("location == 'uw'").reset_index(drop=True)
df_shac_mimic = df_shac.query("location == 'mimic'").reset_index(drop=True)


In [3]:
print(df_dynGen['label_binary'].sum()/len(df_dynGen))
print(df_wsf['label_binary'].sum()/len(df_wsf))

0.5389607233132413
0.1117443707371765


In [6]:
print(df_shac_uw['Drug'].sum()/len(df_shac_uw))
print(df_shac_mimic['Drug'].sum()/len(df_shac_mimic))

0.41139240506329117
0.1976558337773042


In [14]:
## Hate Speech
df0 = df_dynGen
df1 = df_wsf
df_split_label = "label_binary"

p_pos_train_z0_ls = np.arange(0, 1, 0.1) # probability of training set examples drawn from site/domain z0 being positive
p_pos_train_z1_ls = np.arange(0, 1, 0.1) # probability of test set examples drawn from site/domain z1 being positive
p_mix_z1_ls     = np.arange(0.1, 0.9, 0.1) 
n_test = 1000



## SHAC

# df_split_label = "Drug"

# df0 = df_shac_uw
# df1 = df_shac_mimic
# p_pos_train_z0_ls = np.arange(0, 1, 0.1)
# p_pos_train_z1_ls = np.arange(0, 1, 0.1)
# p_mix_z1_ls = np.arange(0, 1, 0.05)

# n_test = 200

##### Split

train_test_ratio = 4


numvals = 1023
base = 1.1
alpha_test_ls = np.power(base, np.arange(numvals)) / np.power(base, numvals // 2)



valid_full_settings = []
for combination in itertools.product(
    p_pos_train_z0_ls, p_pos_train_z1_ls, p_mix_z1_ls, alpha_test_ls
):
    number_setting = number_split(
        p_pos_train_z0=combination[0],
        p_pos_train_z1=combination[1],
        p_mix_z1=combination[2],
        alpha_test=combination[3],
        train_test_ratio=train_test_ratio,
        n_test=n_test,
        verbose=False,
    )

    
            
    if number_setting is not None:
        if np.all([number_setting[k] >= 10 for k in list(number_setting.keys())[:-1]]):
            valid_full_settings.append(number_setting)





valid_n_full_settings = []

for c in tqdm(valid_full_settings):
        c = c.copy()
        # create train/test split according to stats
        dfs = create_mix(df0=df0, df1=df1, target=df_split_label, setting=c, sample=False, 
                         seed=222
                        )

        if dfs is None:
            continue
        
        valid_n_full_settings.append(c)

/home/NETID/xiruod/projects/DeconDTN/notebooks_xiruo/../src/utils.py:29: RuntimeWarning: invalid value encountered in scalar divide
  alpha_train = p_pos_train_z1 / p_pos_train_z0
/home/NETID/xiruod/projects/DeconDTN/notebooks_xiruo/../src/utils.py:29: RuntimeWarning: divide by zero encountered in scalar divide
  alpha_train = p_pos_train_z1 / p_pos_train_z0


  0%|          | 0/27937 [00:00<?, ?it/s]

/home/NETID/xiruod/projects/DeconDTN/notebooks_xiruo/../src/utils.py:263: UserWarning: Set sample equals to True or augment current dataset.
  warnings.warn("Set sample equals to True or augment current dataset.")
/home/NETID/xiruod/projects/DeconDTN/notebooks_xiruo/../src/utils.py:263: UserWarning: Set sample equals to True or augment current dataset.
  warnings.warn("Set sample equals to True or augment current dataset.")
/home/NETID/xiruod/projects/DeconDTN/notebooks_xiruo/../src/utils.py:263: UserWarning: Set sample equals to True or augment current dataset.
  warnings.warn("Set sample equals to True or augment current dataset.")
/home/NETID/xiruod/projects/DeconDTN/notebooks_xiruo/../src/utils.py:263: UserWarning: Set sample equals to True or augment current dataset.
  warnings.warn("Set sample equals to True or augment current dataset.")
/home/NETID/xiruod/projects/DeconDTN/notebooks_xiruo/../src/utils.py:263: UserWarning: Set sample equals to True or augment current dataset.
  w

In [15]:
tmp = [x['mix_param_dict'] for x in valid_n_full_settings]

In [16]:
df = pd.DataFrame(tmp)

In [23]:
df.query("(alpha_train == 0.2) and (C_z == 0.5) ")

,p_pos_train_z0,p_pos_train_z1,p_pos_train,p_pos_test,p_mix_z0,p_mix_z1,alpha_train,alpha_test,p_pos_test_z0,p_pos_test_z1,C_y,C_z
9870,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.035584,0.579383,0.020617,0.3,0.5
9871,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.039143,0.577399,0.022601,0.3,0.5
9872,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.043057,0.575232,0.024768,0.3,0.5
9873,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.047362,0.572868,0.027132,0.3,0.5
9874,0.5,0.1,0.3,0.3,0.5,0.5,0.2,0.052099,0.570289,0.029711,0.3,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...
9936,0.5,0.1,0.3,0.3,0.5,0.5,0.2,19.194342,0.029711,0.570289,0.3,0.5
9937,0.5,0.1,0.3,0.3,0.5,0.5,0.2,21.113777,0.027132,0.572868,0.3,0.5
9938,0.5,0.1,0.3,0.3,0.5,0.5,0.2,23.225154,0.024768,0.575232,0.3,0.5
9939,0.5,0.1,0.3,0.3,0.5,0.5,0.2,25.547670,0.022601,0.577399,0.3,0.5


# HateSpeech

In [18]:
valid_n_full_settings[6126]

{'n_train': 4000,
 'n_test': 1000,
 'n_z0_pos_train': 600,
 'n_z0_neg_train': 1400,
 'n_z0_pos_test': 150,
 'n_z0_neg_test': 350,
 'n_z1_pos_train': 600,
 'n_z1_neg_train': 1400,
 'n_z1_pos_test': 150,
 'n_z1_neg_test': 350,
 'mix_param_dict': {'p_pos_train_z0': 0.30000000000000004,
  'p_pos_train_z1': 0.30000000000000004,
  'p_pos_train': 0.30000000000000004,
  'p_pos_test': 0.30000000000000004,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 1.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.30000000000000004,
  'p_pos_test_z1': 0.30000000000000004,
  'C_y': 0.30000000000000004,
  'C_z': 0.5}}

In [24]:
valid_n_full_settings[9870]

{'n_train': 4000,
 'n_test': 1000,
 'n_z0_pos_train': 1000,
 'n_z0_neg_train': 1000,
 'n_z0_pos_test': 290,
 'n_z0_neg_test': 210,
 'n_z1_pos_train': 200,
 'n_z1_neg_train': 1800,
 'n_z1_pos_test': 10,
 'n_z1_neg_test': 490,
 'mix_param_dict': {'p_pos_train_z0': 0.5,
  'p_pos_train_z1': 0.1,
  'p_pos_train': 0.3,
  'p_pos_test': 0.3,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 0.2,
  'alpha_test': 0.035584102738367304,
  'p_pos_test_z0': 0.5793831697622975,
  'p_pos_test_z1': 0.0206168302377025,
  'C_y': 0.3,
  'C_z': 0.5}}

# SHAC

In [13]:
valid_n_full_settings[2800]

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 80,
 'n_z0_neg_train': 320,
 'n_z0_pos_test': 20,
 'n_z0_neg_test': 80,
 'n_z1_pos_train': 80,
 'n_z1_neg_train': 320,
 'n_z1_pos_test': 20,
 'n_z1_neg_test': 80,
 'mix_param_dict': {'p_pos_train_z0': 0.2,
  'p_pos_train_z1': 0.2,
  'p_pos_train': 0.2,
  'p_pos_test': 0.2,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 1.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.2,
  'p_pos_test_z1': 0.2,
  'C_y': 0.2,
  'C_z': 0.5}}